In [ ]:
!pip install ta-lib
!pip install gdown
!pip install requests
!pip install numpyS
!pip install pandas


In [ ]:
!pip install -r requirements_dev.txt       
!pip install pyotp
!pip install logzero
!pip install websocket-client    

In [ ]:
!pip uninstall pycrypto
!pip install pycryptodome    

In [24]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime
import socket
import uuid
import http.client
import time
import requests # type: ignore
import mimetypes
import json
import talib
import gdown
import http
import ssl
import os

In [25]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [26]:
from SmartApi import SmartConnect #or from SmartApi.smartConnect import SmartConnect
import pyotp
from logzero import logger
from dotenv import load_dotenv

In [27]:
load_dotenv()

True

In [ ]:
# Static values
user_type = "USER"
source_id = "WEB"
api_key = os.getenv("ANG_ONE_KEY")   
client_code = os.getenv("CLIENTCODE")
password = os.getenv("PASSWORD")
window = 365

# todays_date = datetime.today().strftime("%Y-%m-%d")
todays_date = (datetime.today() - pd.DateOffset(days=0)).strftime("%Y-%m-%d")
window_date = (datetime.today() - pd.DateOffset(days=window)).strftime("%Y-%m-%d")

In [46]:
local_ip = socket.gethostbyname(socket.gethostname())
smartApi = SmartConnect(api_key)

try:
    token = "QYO6BHBHL2LLY42CDR7CVIN5SU"
    totp = pyotp.TOTP(token).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e


# Get Public IP
public_ip = requests.get('https://api.ipify.org').text

# Get MAC Address
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])

# Change clientcode, password, totp
payload = '''{\n\"clientcode\":\"'''+str(client_code)+'''\"
         ,\n\"password\":\"'''+str(password)+'''\"\n
		,\n\"totp\":\"'''+str(totp)+'''\"\n
    ,\n\"state\":\"Active\"\n}'''

headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key #'QNeuDKb5'
}


context = ssl._create_unverified_context()

conn = http.client.HTTPSConnection(
    "apiconnect.angelone.in", context=context
    )

conn.request("POST", "/rest/auth/angelbroking/user/v1/loginByPassword", payload, headers)

res = conn.getresponse()
data = res.read()
data = data.decode("utf-8")
# print(data)

[I 251026 19:42:11 smartConnect:121] in pool


In [30]:
temp = json.loads(data)
jwtToken = temp["data"]["jwtToken"]
print(jwtToken)

# user_type = "USER"
# source_id = "WEB"
local_ip = socket.gethostbyname(socket.gethostname())
public_ip = requests.get('https://api.ipify.org').text
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])
authToken = f'Bearer {jwtToken}'


headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key,
    'Authorization': authToken ,
}


eyJhbGciOiJIUzUxMiJ9.eyJ1c2VybmFtZSI6IkJHQkcxMTQ0Iiwicm9sZXMiOjAsInVzZXJ0eXBlIjoiVVNFUiIsInRva2VuIjoiZXlKaGJHY2lPaUpTVXpJMU5pSXNJblI1Y0NJNklrcFhWQ0o5LmV5SjFjMlZ5WDNSNWNHVWlPaUpqYkdsbGJuUWlMQ0owYjJ0bGJsOTBlWEJsSWpvaWRISmhaR1ZmWVdOalpYTnpYM1J2YTJWdUlpd2laMjFmYVdRaU9qRXhMQ0p6YjNWeVkyVWlPaUl6SWl3aVpHVjJhV05sWDJsa0lqb2lZMlZrWkRreU9XWXRaV1ZsWkMwek1ERmlMV0k1TldVdE9EZ3lZVEk1TkdVM01EQTFJaXdpYTJsa0lqb2lkSEpoWkdWZmEyVjVYM1l5SWl3aWIyMXVaVzFoYm1GblpYSnBaQ0k2TVRFc0luQnliMlIxWTNSeklqcDdJbVJsYldGMElqcDdJbk4wWVhSMWN5STZJbUZqZEdsMlpTSjlMQ0p0WmlJNmV5SnpkR0YwZFhNaU9pSmhZM1JwZG1VaWZYMHNJbWx6Y3lJNkluUnlZV1JsWDJ4dloybHVYM05sY25acFkyVWlMQ0p6ZFdJaU9pSkNSMEpITVRFME5DSXNJbVY0Y0NJNk1UYzJNVFU1TVRjMU1pd2libUptSWpveE56WXhOVEExTVRjeUxDSnBZWFFpT2pFM05qRTFNRFV4TnpJc0ltcDBhU0k2SWpneU5EZzJNakl4TFRVME1XTXROR1JtWVMxaFkyRTBMV00wTTJaa05EWTBZbUk0WXlJc0lsUnZhMlZ1SWpvaUluMC5oU2dzbFJDNzhTQnFBMHUtQWNpUk5VUmY0dE1pUHAyNFR5Ym5DcTVPNDNoZW9tbzYweExSY0JFTngwQXIzX29qTTVScWw3MjJPb0V5Y1F5X2F6OWxnb1dRSl92eU9uOTE5V09oM0JSZXByYVBmVVhEZW80TlZ

In [31]:
# shareable_link = 'https://drive.google.com/file/d/1PdYMxjWQ4tBJp4Mmp1LjLOR2H2vkZ6on/view?usp=sharing'

# Extract the file ID
# file_id = shareable_link.split('/d/')[1].split('/view')[0]

# Construct the download URL
# download_url = f'https://drive.google.com/uc?id={file_id}'


# # Download the file using gdown
# output_file = 'Nifty500-token.csv'

# stock_symbols_df = pd.read_csv(output_file)
# stock_symbols_df["token"] = stock_symbols_df["token"].fillna(891)
# stock_symbols_df["token"] = stock_symbols_df["token"].astype(int)
# stocks_to_consider = 'PO1_Stocks.csv'

# stock_lists = pd.read_csv(stocks_to_consider)
# main_df = pd.merge(stock_lists[['rsi','symbol','win_ratio','priority']], stock_symbols_df[['Symbol','token']], left_on ='symbol' , right_on='Symbol', how='inner')

# main_df.to_csv('Main_df.csv', index=False)

In [42]:

main_df = pd.read_csv('Main_df.csv')


In [41]:
# print(main_df)

In [34]:
# Step 2: Function to fetch daily candle data from API
def fetch_candle_data(symbol):
  
    payload = '''{\r\n     \"exchange\": \"NSE\",\r\n
          \"symboltoken\": \"'''+str(symbol)+'''\",\r\n     \"interval\": \"ONE_DAY\",\r\n
          \"fromdate\": \"'''+str(window_date)+''' 16:30\",\r\n     \"todate\": \"'''+str(todays_date)+''' 16:30\"\r\n}
    '''
    # payload = '''{\r\n     \"exchange\": \"NSE\",\r\n
    #       \"symboltoken\": \"'''+str(symbol)+'''\",\r\n     \"interval\": \"ONE_DAY\",\r\n
    #       \"fromdate\": \"2025-09-01 16:30\",\r\n     \"todate\": \"2025-12-31 16:30\"\r\n}
    # '''

    conn = http.client.HTTPSConnection("apiconnect.angelone.in", context=context)
    conn.request("POST", "/rest/secure/angelbroking/historical/v1/getCandleData", payload, headers)
    res = conn.getresponse()
    data = res.read()
    data = data.decode("utf-8")
    json_data = json.loads(data)
    json_data = json_data['data']
    # print(json_data)
    return json_data

# Step 4: Function to calculate RSI trends (increase or decrease)
def rsi_trend1(rsi_values):
    if rsi_values[-1] >= 60 and  rsi_values[-2] < 60:
      return "60 CROSSOVER"

    if rsi_values[-1] >= 40 and  rsi_values[-2] < 40:
      return "ON"

    if rsi_values[-1] < 40 and rsi_values[-2] >= 40:
      return "EXIT"

    if rsi_values[-1] > rsi_values[-2]:
        return 'UP'
    else:
        return 'DOWN'
    

# Fuction to identify RSI Breakout
def rsi_trend(rsi_values, setup_rsi):
    if rsi_values[-1] >= setup_rsi and  rsi_values[-2] < setup_rsi:
      return True
    

# Fuction to identify stocks before RSI Breakout 
def rsi_trend_P1(rsi_values, setup_rsi):    
    if rsi_values[-1] >= ( setup_rsi - 5 ) and  rsi_values[-1] < ( setup_rsi + 5):
      return True
    

In [43]:
# Prepare output DataFrame
output_data = []
error_data = []
priority_data = []
priority0_data = []

print(main_df.columns.tolist())

# Step 5: Process each stock
for _, row in main_df.iterrows():

    time.sleep(0.4)

    # company = row['NAME OF COMPANY']
    priority = row['priority']
    name = row['Symbol']
    token = row['token']
    rsi = row['rsi']
    win_ratio = row['win_ratio']
    try:
        # rsi_value = setup_df.loc[setup_df["symbol"] == name, "rsi"].iloc[0]
        # rsi_value = setup_df.loc[setup_df["symbol"] == name, "rsi"].iloc[0] if not setup_df.loc[setup_df["symbol"] == name, "rsi"].empty else None
        # print(f"RSI of {name}: {rsi}")
        if rsi == None:
            continue
   
   
        # Fetch daily data
        daily_json_data = fetch_candle_data(token)
        if daily_json_data == None:
            error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'Error while fetching data'
             })    
            continue
        # print(daily_json_data)
        df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
        
        # break

        # Convert 'Date' column to datetime
        df['Date'] = pd.to_datetime(df['Date'])

        # Sort data by date in ascending order
        df = df.sort_values('Date').reset_index(drop=True)
        df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)
        # df['RSI_14_cal'] = rsi_calc(df['Close'], period=14)
        df.set_index('Date', inplace=True)
        
        
        
        last_2_rsi_daily = df['RSI_14'].dropna().tail(2).values
        # last_2_rsi_daily_dates = df['Date']

        

        
        if len(last_2_rsi_daily) < 2:
             error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': 'Not enough RSI data'
             })
             continue
        daily_rsi = last_2_rsi_daily[-1]

        last_dats = daily_json_data[-1][0].split('T')[0]
        # print('last date',last_dats)
        if last_dats != todays_date:
             error_data.append({
               'Name': name,
               'Token': token,
               'Setup_RSI': rsi,
               'reasone': f'Date from API {last_dats}, processing date {todays_date}'
             })
             continue


        if priority == 2 and rsi_trend_P1(last_2_rsi_daily, rsi):
            priority_data.append({
                'Name': name,
                'Token': token,
                'Setup_RSI': rsi,
                'Daily_RSI': daily_rsi,
                'yesterday_RSI': last_2_rsi_daily[-2],
                'win_ratio': win_ratio,
                'RSI_Trend': rsi_trend_P1(last_2_rsi_daily, rsi),

            })
        elif priority == 1 and rsi_trend(last_2_rsi_daily, rsi):
            output_data.append({
                'Name': name,
                'Token': token,
                'Setup_RSI': rsi,
                'Daily_RSI': daily_rsi,
                'yesterday_RSI': last_2_rsi_daily[-2],
                'win_ratio': win_ratio,

            })
        elif priority == 0 and rsi_trend(last_2_rsi_daily, rsi):
            priority0_data.append({
                'Name': name,
                'Token': token,
                'Setup_RSI': rsi,
                'Daily_RSI': daily_rsi,
                'yesterday_RSI': last_2_rsi_daily[-2],
                'win_ratio': win_ratio,

            })
            
        
            
        
        # print(name, daily_rsi, last_2_rsi_daily[-2], token)

        # Add the data to the output with Company and Token
        

    except Exception as e:
        error_data.append({
            'Name': name,
            'Token': token,
            'Setup_RSI': rsi,
            'reasone': str(e)
        })
        print(f"Error processing {name}: {e}")

['rsi', 'symbol', 'win_ratio', 'priority', 'Symbol', 'token']


In [36]:
print("Priority Data:")
print(priority_data)    
print("Output Data:")
print(output_data)  
print("Error Data:")
print(error_data)

Priority Data:
[{'Name': 'KFINTECH', 'Token': 13364, 'Setup_RSI': 59, 'Daily_RSI': np.float64(57.35908809168904), 'yesterday_RSI': np.float64(54.6899366935105), 'win_ratio': '63.16%', 'RSI_Trend': True}, {'Name': 'ICICIBANK', 'Token': 12458, 'Setup_RSI': 52, 'Daily_RSI': np.float64(55.376474066559375), 'yesterday_RSI': np.float64(48.28291553203174), 'win_ratio': '65.06%', 'RSI_Trend': True}, {'Name': 'KAYNES', 'Token': 12092, 'Setup_RSI': 49, 'Daily_RSI': np.float64(47.50125985134672), 'yesterday_RSI': np.float64(45.974580687129276), 'win_ratio': '75.61%', 'RSI_Trend': True}, {'Name': 'BAJAJHLDNG', 'Token': 305, 'Setup_RSI': 42, 'Daily_RSI': np.float64(41.22834875891477), 'yesterday_RSI': np.float64(41.06816316553289), 'win_ratio': '64.00%', 'RSI_Trend': True}]
Output Data:
[{'Name': 'M&M', 'Token': 2031, 'Setup_RSI': 50, 'Daily_RSI': np.float64(53.04314925067107), 'yesterday_RSI': np.float64(48.932467298572554), 'win_ratio': '56.32%'}, {'Name': 'ULTRACEMCO', 'Token': 11532, 'Setup_RSI

In [44]:
# Create Files For the output

todays_date = datetime.today().strftime("%Y-%m-%d")
output_df = pd.DataFrame(output_data)
output_df.to_csv(f'Daily_Report/Trending/RSI-Setup-treading-{todays_date}.csv', index=False)
error_df = pd.DataFrame(error_data)
error_df.to_csv(f'Daily_Report/Error/RSI-Setup-error-{todays_date}.csv', index=False)
priority_df = pd.DataFrame(priority_data)
priority_df.to_csv(f'Daily_Report/Priority/RSI-Setup-Priority-{todays_date}.csv', index=False)

In [38]:
import requests
# import urllib.parse

BOT_TOKEN = "8446280700:AAEVJcAw73988-gAx8kJF1TKFMLwHVCM-gs"

TEST_ID = "529251493"
CHAT_ID = TEST_ID
# CHAT_ID = "-1003139839259"
def format_whatsapp_report(data ,name):
    
    lines = [f"📊 <b>{name}</b>"]
    
    if len(data) == 0:
        lines.append( "\n🔹 <b>No Stocks</b>" )
    else:
        for index,item in enumerate(data):
            lines.append(
                f"\n🔹 <b>{index+1} {item['Name']}</b>"
                f"\n   <b>Todays_RSI: </b>{item['Daily_RSI']:.2f}"
                f"\n   <b>Yesterdays_RSI: </b>{item['yesterday_RSI']:.2f}"
                f"\n   <b>Standard_RSI: </b>{item['Setup_RSI']:.2f}"
                # f"\n   High Priority: {'✅' if item['High Priority'] else '❌'}"
            )      
           
    # return urllib.parse.quote_plus( "\n".join(lines) )
    return "\n".join(lines) 

def format_whatsapp_error(data ,name):
    
    lines = [f"📊 <b>{name}</b>"]
    
    if len(data) == 0:
        lines.append( "\n🔹 <b>No Stocks</b>" )
    else:
        for index,item in enumerate(data):
            lines.append(
                f"\n🔹 <b>{index+1} {item['Name']}</b>"
                f"\n   <b>Token: </b>{item['Token']}"
                f"\n   <b>Standard_RSI: </b>{item['Setup_RSI']:.2f}"
                f"\n   <b>Reasone: </b>{item['reasone']}"
                # f"\n   High Priority: {'✅' if item['High Priority'] else '❌'}"
            )      
           
    # return urllib.parse.quote_plus( "\n".join(lines) )
    return "\n".join(lines)

In [39]:
# Telegram Message trigger logic
msg = f"📊 <b>Daily Report: {todays_date}</b>"
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
             params={"chat_id": CHAT_ID, "text": msg, "parse_mode": "HTML"})


msg_p1 = format_whatsapp_report(priority_data,'Priority Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
             params={"chat_id": CHAT_ID, "text": msg_p1, "parse_mode": "HTML"})


msg_t = format_whatsapp_report(output_data ,'Treading Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
             params={"chat_id": CHAT_ID, "text": msg_t, "parse_mode": "HTML"})


msg_p0 = format_whatsapp_report(priority0_data,'Least Priority Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
            params={"chat_id": CHAT_ID, "text": msg_p0, "parse_mode": "HTML"})



<Response [200]>

In [40]:
error_msg = format_whatsapp_error(error_data,'Error Stocks')
requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
            params={"chat_id": TEST_ID, "text": error_msg, "parse_mode": "HTML"})

<Response [200]>